# Notatbøker og tensorer

**Du skal kunne:** kjøre celler i riktig rekkefølge, lese dimensjoner og forklare
hvorfor GPU først hjelper når det er nok arbeid. Her trenger du ingen nedlastede data.

En notatbok er en arbeidsbok med tekst og kjørbare kodeceller. Teksten forklarer forsøket;
resultatet står under koden. Det er nyttig når vi vil se bilde, prediksjon og graf sammen.
Senere legger vi gjenbrukbar kode i vanlige Python-filer.

Klikk i en kodecelle og trykk **Shift+Enter**. En **kernel** er Python-prosessen
som husker variablene. Den husker også gamle verdier hvis du endrer en celle uten å
kjøre den igjen. Bruk **Restart Kernel and Run All** før du stoler på et resultat.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Tall med en form

En tensor er en beholder for tall med en form, omtrent som en eggekartong.
En vektor har én akse; en tabell har to. Et bilde kan ha kanal, høyde og bredde.
En batch er flere eksempler samlet i én leveranse. Python teller fra null.

`torch.tensor` lager tallene, `.shape` viser formen og `[0]` henter første rad.
`dtype` er talltypen: desimaltall til beregninger og heltall til kategorier.
**Forutsi:** Hvor mange tall har tabellen, og hva blir første rad?


In [ ]:
tall = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print(tall)
print("Form:", tall.shape, "type:", tall.dtype)
print("Første rad:", tall[0])
print("Alle tall pluss ti:", tall + 10)


## Fra bilde til MLP

En **MLP** er et nettverk med fullt sammenkoblede lag. Det venter en liste med tall
per eksempel. `.flatten(start_dim=1)` beholder batchaksen og flater ut resten.
Som å legge radene i et ruteark etter hverandre på et langt bånd.
`zeros` lager bare nuller; dette er kunstige bilder for å forstå formen.

**Forutsi:** Fire gråtonebilder på 28 × 28 blir fire lister med hvor mange tall?


In [ ]:
bilder = torch.zeros(4, 1, 28, 28)
flate_bilder = bilder.flatten(start_dim=1)
print("Bilder [batch, kanal, høyde, bredde]:", bilder.shape)
print("MLP-input [batch, piksler]:", flate_bilder.shape)


## Send tall til GPU

`.to(enhet)` flytter tallene. `@` betyr matrisemultiplikasjon: hver rad kombineres
med hver kolonne. Mange slike uavhengige beregninger passer en GPU.
GPU-en er som et stort kjøkken med mange kokker; små bestillinger kan bruke mer tid
på organisering enn på matlaging. Vi må måle, ikke anta at alt blir raskere.

**Forutsi:** Hvilken form får `c` når to 256 × 256-matriser multipliseres?


In [ ]:
a = torch.randn(256, 256).to(enhet)
b = torch.randn(256, 256).to(enhet)
c = a @ b
print(c.shape, c.device)


## Prøv selv og kontroller forståelsen

1. Endre antall bilder fra 4 til 7. Hvilken akse endres?
2. Lag RGB-bilder med tre kanaler. Hvor mange tall får MLP-en nå?
3. Kjør siste celle etter en kernel-restart uten å kjøre de første. Forklar feilen,
   og velg deretter Restart Kernel and Run All.

**Fasit:** 784 tall per gråtonebilde; 2352 for RGB. Batchaksen er først.
En `NameError` etter restart betyr ofte at cellen som definerer variabelen ikke er kjørt.

**Neste:** [01-gradienter-og-laering.ipynb](01-gradienter-og-laering.ipynb).
